# Superstore Sales & Profitability Analysis

**Tools:** Python, Pandas

## Project Overview
This notebook performs exploratory data analysis on the Superstore sales dataset. The analysis focuses on data quality, sales and profitability trends, discount levels, product categories, and key business KPIs.

The cleaned dataset produced here is subsequently used in the PostgreSQL/SQL and Power BI stages of the end-to-end portfolio project.


## 1. Import Libraries and Load Data

The dataset is loaded into a Pandas DataFrame. For GitHub portability, the notebook expects the dataset to be placed in the repository's `data` folder rather than using a personal computer file path.


In [1]:
from pathlib import Path
import csv
import pandas as pd

DATA_PATH = Path("../data/Superstore new.csv")

df = pd.read_csv(
    DATA_PATH,
    encoding="latin1"
)


In [2]:
# Preview the first five rows
df.head()


## 2. Dataset Structure and Data Quality

Before analysis, the dataset is checked for its dimensions, column names, data types, duplicates, missing values, and date consistency.


In [3]:
# Number of rows and columns
df.shape


In [4]:
# Column names
df.columns


In [5]:
# Data types and non-null counts
df.info()


In [6]:
# Check for completely duplicated rows
df.duplicated().sum()


In [7]:
# Check whether Row ID is duplicated
df["Row ID"].duplicated().sum()


In [8]:
# Check for missing values
df.isnull().sum()


### Date Validation

The order and shipping dates are converted to datetime format, then checked to ensure no shipment date occurs before the corresponding order date.


In [9]:
# Convert date columns to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])


In [10]:
# Confirm the converted data types
df[["Order Date", "Ship Date"]].dtypes


In [11]:
# Check for invalid date relationships
(df["Ship Date"] < df["Order Date"]).sum()


## 3. Descriptive Statistics

The numerical variables are summarized to understand their distributions and identify the range of sales, quantity, discounts, and profit.


In [12]:
df[["Sales", "Quantity", "Discount", "Profit"]].describe()


In [13]:
# Number and percentage of records with negative profit
negative_profit_count = (df["Profit"] < 0).sum()
negative_profit_pct = (df["Profit"] < 0).mean() * 100

negative_profit_count, negative_profit_pct


## 4. Dataset Dimensions and Categories

The dataset contains several categorical dimensions used later for segmentation and dashboard analysis.


In [14]:
print("Ship Modes:", df["Ship Mode"].unique())
print("Segments:", df["Segment"].unique())
print("Regions:", df["Region"].unique())
print("Categories:", df["Category"].unique())
print("Countries:", df["Country"].unique())
print("Sub-Categories:", df["Sub-Category"].unique())
print("Number of States:", df["State"].nunique())


In [15]:
# Unique orders, customers, and products
df[["Order ID", "Customer ID", "Product ID"]].nunique()


## 5. Key Business KPIs

These calculations provide an overall view of sales, profit, units sold, profitability, and order value.


In [16]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_quantity = df["Quantity"].sum()
profit_margin = total_profit / total_sales * 100
average_order_value = total_sales / df["Order ID"].nunique()
average_profit_per_order = total_profit / df["Order ID"].nunique()

kpis = pd.Series({
    "Total Sales": total_sales,
    "Total Profit": total_profit,
    "Total Quantity Sold": total_quantity,
    "Profit Margin (%)": profit_margin,
    "Average Order Value": average_order_value,
    "Average Profit per Order": average_profit_per_order
})

kpis


## 6. Sales and Profit by Year

The analysis groups sales and profit by year and calculates annual profit margin. Negative-profit transactions are retained so the results represent overall net performance.


In [17]:
# Extract year from Order Date
df["Year"] = df["Order Date"].dt.year

df["Year"].unique()


In [18]:
yearly_performance = (
    df.groupby("Year")[["Sales", "Profit"]]
      .sum()
)

yearly_performance["Profit Margin %"] = (
    yearly_performance["Profit"] /
    yearly_performance["Sales"] * 100
)

yearly_performance


## 7. Discount and Profitability Analysis

Profitability is examined across discount levels. This analysis describes the association between discount levels and profitability; it does not by itself establish causation.


In [19]:
discount_analysis = (
    df.groupby("Discount")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Transactions=("Profit", "count")
      )
)

discount_analysis["Profit Margin %"] = (
    discount_analysis["Profit"] /
    discount_analysis["Sales"] * 100
)

discount_analysis.sort_index()


## 8. Category-Level Performance

Sales, profit, average discount, transaction count, and profit margin are compared across product categories.


In [20]:
category_discount = (
    df.groupby("Category")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Avg_Discount=("Discount", "mean"),
          Transactions=("Profit", "count")
      )
)

category_discount["Profit Margin %"] = (
    category_discount["Profit"] /
    category_discount["Sales"] * 100
)

category_discount.sort_values("Profit", ascending=False)


## 9. Prepare the Cleaned Dataset

A separate DataFrame is created so the original loaded DataFrame remains available. The cleaned version includes the converted date fields and the derived `Year` column.

The cleaned CSV is then saved for use in the PostgreSQL stage of the project.


In [21]:
df_clean = df.copy()

df_clean.info()


In [22]:
# Verify the key date and year fields
df_clean[["Order Date", "Ship Date", "Year"]].head()


In [23]:
# Final missing-value check
df_clean.isnull().sum()


In [24]:
# Save cleaned dataset
df_clean.to_csv("superstore_clean.csv", index=False)


In [25]:
# Confirm the cleaned file was created
Path("superstore_clean.csv").exists()


In [26]:
# Check file size
Path("superstore_clean.csv").stat().st_size


### PostgreSQL-Compatible Export

A second export is created with standard CSV quoting for the PostgreSQL import stage.


In [27]:
df_clean.to_csv(
    "superstore_clean_pg.csv",
    index=False,
    quoting=csv.QUOTE_MINIMAL
)


In [28]:
# Validate the exported PostgreSQL CSV
test_df = pd.read_csv("superstore_clean_pg.csv")

print("Shape:", test_df.shape)
print("Missing values:", test_df.isnull().sum().sum())


## 10. Summary

The Python stage validates the dataset, performs the initial exploratory analysis, calculates core business metrics, and produces the cleaned CSV used in the next stages of the project.

The analysis is continued in **PostgreSQL/SQL** and then presented through an interactive **Power BI** dashboard.
